In [1]:
!python --version
import os
import warnings
print (os.environ['CONDA_DEFAULT_ENV'])

#suppress warnings
warnings.filterwarnings('ignore')

Python 3.9.21
stan


In [2]:
import pandas as pd
import numpy as np
import xarray as xr
xr.set_options(file_cache_maxsize=10)

import scipy as scipy
from scipy import stats
import math
from scipy.stats import pearsonr, spearmanr, truncnorm
from scipy.stats import gaussian_kde
from sklearn.linear_model import LinearRegression, RANSACRegressor, HuberRegressor, TheilSenRegressor

import statsmodels.api as sm

import matplotlib as mpl

import matplotlib.pyplot as plt
import matplotlib.colors as colors
import cmocean.cm as cmo

import proplot as plot
import seaborn as sns
from shapely.geometry import Point, Polygon

from geopy.distance import distance, Distance, lonlat

from matplotlib import font_manager
font_manager.findfont("TeX Gyre Heros")
mpl.rcParams.update({'font.sans-serif':'TeX Gyre Heros',
                    'font.weight': 'normal',
                    'axes.labelweight': 'normal',
                    'axes.titleweight': 'normal',
                    'pdf.fonttype':42,
                    'ps.fonttype':42
                     })

import string
import os
import requests
import io
from tqdm import tqdm
import time


In [3]:
def round_to_nearest_half_int(num):
    return round(num * 2) / 2

def getClosestPoint_fromLineString(LineString_input,point1):
    if "MultiLineString" in str(type(LineString_input)):
        multiLineString = LineString_input
        len_multiLineString = len(multiLineString.geoms)
        idxMin = np.array([multiLineString.geoms[i].distance(point1) for i in range(len_multiLineString)]).argmin()
        
        LineString = multiLineString.geoms[idxMin]
    elif "Polygon" in str(type(LineString_input)):
        
        polygon_input = LineString_input
        LineString = polygon_input.exterior
        
    else:
        LineString = LineString_input
        
    Coords = LineString.coords
    x, y = Coords.xy

    coords_df = pd.DataFrame({'LON':x,'LAT':y})
    coords_df['LAT_diff'] = abs(point1.y-coords_df.LAT)
    coords_df['LON_diff'] = abs(point1.x-coords_df.LON)
    coords_df['LONLAT_diff_sum'] = coords_df['LAT_diff']+coords_df['LON_diff']
    lat2,lon2 = coords_df.LAT.iloc[coords_df.LONLAT_diff_sum.idxmin()],coords_df.LON.iloc[coords_df.LONLAT_diff_sum.idxmin()]
    point2 = Point(lon2,lat2)
    return point2

def distance_haversine(origin, destination, output_unit='km'):
    import math
    lon1, lat1 = origin
    lon2, lat2 = destination
    
    if output_unit=='km':
        radius = 6371
    elif output_unit=='mi':
        radius = 3956
    elif output_unit=='m':
        radius = 6371000
    
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) * math.sin(dlat / 2) +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(dlon / 2) * math.sin(dlon / 2))
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    d = radius * c
    return d

# Define usefull functions:
def vrange(V): # Return the array value range
    return [np.min(V),np.max(V)]
def vrangec(V): # Return the array value centered range 
    xl = np.max(np.abs(vrange(V)))
    return np.array([-xl,xl])

def nonnan_gradient(array, axis=None, edge_order=1):
  """Computes the gradient of an array, ignoring NaN values.

  Args:
    array: An N-dimensional array containing samples of a scalar function.
    axis: The axis along which to compute the gradient. If None, the gradient
      is computed for all axes.
    edge_order: The order of the accuracy of the approximation at the edges of
      the array.

  Returns:
    An N-dimensional array or a list of N-dimensional arrays representing the
    gradient.
  """

  # Create a mask of non-NaN values.
  mask = np.isfinite(array)

  # Compute the gradient of the masked array.
  gradient = np.gradient(array, axis=axis, edge_order=edge_order)

  # Set the gradient to zero wherever the mask is False.
  for i in range(len(gradient)):
    gradient[i][~mask] = 0

  return gradient



In [4]:
def predictSSTfromTEX(input_tex,method_name='kim2010_tex86H'):
    '''
    
    available methods:
    
    linear: schouten2002, schouten2003, kim2008, OBrien2017
    non-linear: liu2009, kim2010_tex86H
    '''
    
    reg_params_dict = {
        'schouten2002':(66.6667,-18.6667),
        'schouten2003':(37.0370,0.5926),
        'kim2008':(56.2000,-10.7800),
        'liu2009':(-16.3000,50.4750),
        'kim2010_tex86H':(68.4,38.6),
        'OBrien2017':(58.8235,-11.1765),
        'allCoretop_linear_exclRS_above5degC':(67.63154335,-17.85292574),
        'low23_coretop_lnTEX':(35.5891263830379,41.3154356803057),
        'low23_coretop_linear':(69.61698822,-16.73443208),
        'low23_coretop_cultureAllAOA_lnTEX':(37.32224813,44.59294303),
        'low23_coretop_cultureAllAOA_linear':(65.2193,-14.8708),
        'low23_coretop_cultureMarineAOA_lnTEX':(32.39639713,40.83593007),
        'low23_coretop_cultureMarineAOA_linear':(62.32280847,-13.4871),
    }
    
    if method_name=='kim2010_tex86H':
        predictor = np.log10(input_tex)
    elif method_name=='liu2009':
        predictor = (1/input_tex)
    elif 'lnTEX' in method_name:
        predictor = np.log(input_tex)
    else:
        predictor = input_tex
        
    slope = reg_params_dict.get(method_name)[0]
    yintercept = reg_params_dict.get(method_name)[1]
    
    predictedSST = slope*predictor+yintercept
    return predictedSST

def predictTEXfromSST(input_sst,method_name='kim2010_tex86H'):
    '''
    
    available methods:
    
    linear: schouten2002, schouten2003, kim2008, OBrien2017
    non-linear: liu2009, kim2010_tex86H
    '''
    
    reg_params_dict = {
        'schouten2002':(0.015,0.280),
        'schouten2003':(0.027,-0.016),
        'kim2008':(0.017794,0.191815),
        'liu2009':(-16.3000,50.4750),
        'kim2010_tex86H':(68.4,38.6),
        'OBrien2017':(0.0170,0.19),
        'allCoretop_linear_exclRS_above5degC':(67.63154335,-17.85292574),
        'low23_coretop_lnTEX':(35.5891263830379,41.3154356803057),
        'low23_coretop_linear':(69.61698822,-16.73443208),
        'low23_coretop_cultureAllAOA_lnTEX':(37.32224813,44.59294303),
        'low23_coretop_cultureAllAOA_linear':(65.2193,-14.8708),
        'low23_coretop_cultureMarineAOA_lnTEX':(32.39639713,40.83593007),
        'low23_coretop_cultureMarineAOA_linear':(62.32280847,-13.4871),
    }
    
    if 'lnTEX' in method_name:
        predictor = np.log(input_sst)
    else:
        predictor = input_sst
    
    slope = reg_params_dict.get(method_name)[0]
    yintercept = reg_params_dict.get(method_name)[1]
    if method_name=='liu2009':
        predictedTEX = 1/(slope*predictor+yintercept)
    elif method_name=='kim2010_tex86H':
        predictedTEX = 10**((predictor-yintercept)/slope)
    else:
        predictedTEX = slope*predictor+yintercept
    return predictedTEX

In [5]:
ringNumbers_dict_zhang16 = {
    'reported_1302':0, 
    'fGDGT_0':0,
    'reported_1300':1,
    'fGDGT_1':1, 
    'reported_1298':2,
    'fGDGT_2':2, 
    'reported_1296':3,
    'fGDGT_3':3,
    'reported_1294':4,
    'fGDGT_4':4,
    'reported_1294_iso':4,
    'fGDGT_4_prime':4,
    'reported_1292':4, 
    'fGDGT_cren':4,
    'reported_1292_iso':4,
    'fGDGT_cren_prime':4,
    'reported_1292_iso2':5,
    'fGDGT_5':5,
    'reported_1290':6,
    'fGDGT_6':6,
    'reported_1290_iso':6,
    'fGDGT_6_prime':6,
    'reported_1288':7,
    'fGDGT_7':7,
    'reported_1288_iso':7,
    'fGDGT_7_prime':7,
    'reported_1286':8,
    'fGDGT_8':8,
    'reported_1286_iso':8,
    'fGDGT_8_prime':8,
}

ringNumbers_dict_revised = {
    'reported_1302':0, 
    'fGDGT_0':0,
    'reported_1300':1,
    'fGDGT_1':1, 
    'reported_1298':2,
    'fGDGT_2':2, 
    'reported_1296':3,
    'fGDGT_3':3,
    'reported_1294':4,
    'fGDGT_4':4,
    'reported_1294_iso':4,
    'fGDGT_4_prime':4,
    'reported_1292':5, 
    'fGDGT_cren':5,
    'reported_1292_iso':5,
    'fGDGT_cren_prime':5,
    'reported_1292_iso2':5,
    'fGDGT_5':5,
    'reported_1290':6,
    'fGDGT_6':6,
    'reported_1290_iso':6,
    'fGDGT_6_prime':6,
    'reported_1288':7,
    'fGDGT_7':7,
    'reported_1288_iso':7,
    'fGDGT_7_prime':7,
    'reported_1286':8,
    'fGDGT_8':8,
    'reported_1286_iso':8,
    'fGDGT_8_prime':8,
}


def label_pvalues(x):
    if x<0.001:
        return r'$\it{p}$<.001'
    elif x<0.01:
        return r'$\it{p}$<.01'
    elif x<0.05:
        return r'$\it{p}$<.05'
    elif x<0.1:
        return r'$\it{p}$<.1'
    elif x>=0.1:
        return r'$\it{p}$>.1'


In [6]:
# set the path to local folder other than github
login_name = os.getlogin()
### 1.4.1 Local path on PC
local_documents_path = f'/home/{login_name}/Documents' ### this is the path when code is run on Linux

### 1.4.2 Path to github folder
gitpath_idx = os.getcwd().find('culRI-Bayesian')+len('culRI-Bayesian')
local_github_path = os.getcwd()[:gitpath_idx]

### 1.4.3 Path to OneDrive folder
local_onedrive_path = f'/home/{login_name}/OneDrive'


### stan_models path 
stan_models_path = local_github_path+'/stan_models/'

In [7]:
### Regression dataset compilation
fpath = fr'{local_github_path}/spreadsheets'
fname = r'culture_mesocosm_combined_rev_030425.csv'

culture_meso_df = pd.read_csv(os.path.join(fpath, fname))
culture_meso_growthrate = culture_meso_df[culture_meso_df['experiment_setup'].isin(['Culture-growth-rate','Culture-oxygen'])].reset_index(drop=True)  
culture_meso_df = culture_meso_df[culture_meso_df['experiment_setup'].isin(['mesocosm','Culture-temperature'])].reset_index(drop=True)
culture_meso_df['gdgt23ratio'] = culture_meso_df['fGDGT_2']/culture_meso_df['fGDGT_3']
# culture_meso_df = culture_meso_df[culture_meso_df['Source']!='Sinninghe Damste et al. (2002) Journal of Lipid Research'].reset_index(drop=True)
culture_meso_df['Strains_edited'] = np.nan
for i in range(len(culture_meso_df)):
    if culture_meso_df['Strains'].iloc[i] == 'mesocosm':
        if culture_meso_df['Source'].iloc[i] == 'Wuchter2004':
            culture_meso_df['Strains_edited'].iloc[i] = 'mesocosm - North Sea'
        else:
            culture_meso_df['Strains_edited'].iloc[i] = 'mesocosm - Seychelles'
    else:
        culture_meso_df['Strains_edited'].iloc[i] = culture_meso_df['Strains'].iloc[i]

culture_meso_df['GTDB_genus_edited'] = np.nan
for i in range(len(culture_meso_df)):
    if culture_meso_df['GTDB_genus'].iloc[i] == 'mesocosm':
        if culture_meso_df['Source'].iloc[i] == 'Wuchter2004':
            culture_meso_df['GTDB_genus_edited'].iloc[i] = 'mesocosm - North Sea'
        else:
            culture_meso_df['GTDB_genus_edited'].iloc[i] = 'mesocosm - Seychelles'
    else:
        culture_meso_df['GTDB_genus_edited'].iloc[i] = culture_meso_df['GTDB_genus'].iloc[i]

culture_meso_df = culture_meso_df[culture_meso_df['Source']!='Varma et al. (2024) Biogeosciences']
culture_meso_df


,sampleID,sampleName,Source,Temperature,TEX86,ringIndex,scaledRI,gdgt23ratio,phyloT_order,GTDB_phylum,...,fGDGT_3,fGDGT_cren,fGDGT_cren_prime,datatype,methaneIndex,gdgtZeroOverZeroCren,BIT,violin_bins,Strains_edited,GTDB_genus_edited
0,TEXAS_PSM_culture_RR001,Bale2019_019_Cultures_ThAOA_Ca_Nitrosotenuis u...,Bale et al. (2019) AEM,37,0.578544,2.792553,0.698138,0.478873,13.0,p__Thermoproteota,...,0.125887,0.443262,0.081560,culture,0.420744,0.174917,0,Culture-temperature_Terrestrial thermophile,N4,g__Nitrosotenuis
1,TEXAS_PSM_culture_RR002,Bale2019_020_Cultures_ThAOA_Ca_Nitrosotenuis u...,Bale et al. (2019) AEM,46,0.806931,3.734637,0.933659,0.482759,13.0,p__Thermoproteota,...,0.040503,0.712291,0.167598,culture,0.115169,0.007782,0,Culture-temperature_Terrestrial thermophile,N4,g__Nitrosotenuis
2,TEXAS_PSM_culture_RR003,Bale2019_021_Cultures_ThAOA_Ca_Nitrosotenuis u...,Bale et al. (2019) AEM,50,0.755102,3.725664,0.931416,0.652174,13.0,p__Thermoproteota,...,0.029077,0.745891,0.139064,culture,0.109415,0.008403,0,Culture-temperature_Terrestrial thermophile,N4,g__Nitrosotenuis
3,TEXAS_PSM_culture_RR004,Elling2015_Culture_NA0A2_18,Elling et al. (2015) GCA,18,0.858108,2.437576,0.609394,0.402266,5.2,p__Thermoproteota,...,0.427879,0.161212,0.015758,culture,0.798621,0.429185,0,Culture-temperature_Marine mesophile,NAOA2,g__Nitrosopumilus
4,TEXAS_PSM_culture_RR005,Elling2015_Culture_NA0A2_22,Elling et al. (2015) GCA,22,0.827338,2.235429,0.558857,1.088785,5.2,p__Thermoproteota,...,0.244571,0.200000,0.014857,culture,0.742818,0.451411,0,Culture-temperature_Marine mesophile,NAOA2,g__Nitrosopumilus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,TEXAS_PSM_mesocosm_RR017,meso_017,Schouten2007,25,0.449778,2.772491,0.693123,2.629108,0.0,mesocosm,...,0.013396,0.635200,0.011792,mesocosm,0.159209,0.266255,0,NaN,mesocosm - Seychelles,mesocosm - Seychelles
126,TEXAS_PSM_mesocosm_RR018,meso_018,Schouten2007,32,0.683519,2.627887,0.656972,3.553719,0.0,mesocosm,...,0.015702,0.564495,0.039904,mesocosm,0.169194,0.325581,0,NaN,mesocosm - Seychelles,mesocosm - Seychelles
127,TEXAS_PSM_mesocosm_RR019,meso_019,Schouten2007,34,0.606766,3.236153,0.809038,2.318038,0.0,mesocosm,...,0.019677,0.747221,0.011769,mesocosm,0.131807,0.144080,0,NaN,mesocosm - Seychelles,mesocosm - Seychelles
128,TEXAS_PSM_mesocosm_RR020,meso_020,Schouten2007,36,0.813041,3.598787,0.899697,2.396907,0.0,mesocosm,...,0.022633,0.822493,0.027125,mesocosm,0.106058,0.056856,0,NaN,mesocosm - Seychelles,mesocosm - Seychelles


In [8]:
fpath = f'{local_github_path}/nc_files/'
fname = 'ds06_calculated_ocean_properties.nc'

ocean_properties_ds = xr.open_dataset(os.path.join(fpath, fname))
display(ocean_properties_ds)

fpath = f'{local_github_path}/nc_files/'
fname = 'ds04_gridded_coretop_tex_scaledRI.nc'
coretop_ds = xr.open_dataset(os.path.join(fpath, fname))
coretop_ds

<xarray.Dataset>
Dimensions:        (lat: 720, lon: 1440)
Coordinates:
  * lat            (lat) float32 -89.88 -89.62 -89.38 ... 89.38 89.62 89.88
  * lon            (lon) float32 -179.9 -179.6 -179.4 ... 179.4 179.6 179.9
Data variables:
    no3_sf2tc_avg  (lat, lon) float32 ...
    no3_tc         (lat, lon) float32 ...
    t_sf2tc_avg    (lat, lon) float32 ...
    tc_depth       (lat, lon) float32 ...
    t_tc           (lat, lon) float32 ...
Attributes:
    title:             Calculated Ocean Properties: Nitrite and Temperature f...
    summary:           This dataset contains climatological annual means (199...
    data_sources:      Global Ocean Biogeochemistry Hindcast (CMEMS); World O...
    Conventions:       CF-1.7
    processing_level:  Derived product
    history:           (a) Nitrate: Calculated climatological annual means (1...
    references:        CMEMS, WOA23

<xarray.Dataset>
Dimensions:                (lat: 720, lon: 1440)
Coordinates:
  * lat                    (lat) float32 -89.88 -89.62 -89.38 ... 89.62 89.88
  * lon                    (lon) float32 -179.9 -179.6 -179.4 ... 179.6 179.9
Data variables: (12/14)
    region_ID              (lon, lat) float64 ...
    tex_count              (lon, lat) float64 ...
    tex_median             (lon, lat) float64 ...
    tex_mean               (lon, lat) float64 ...
    tex_std                (lon, lat) float64 ...
    scaledRI_count         (lon, lat) float64 ...
    ...                     ...
    scaledRI_std           (lon, lat) float64 ...
    gdgt23ratio_count      (lon, lat) float64 ...
    gdgt23ratio_median     (lon, lat) float64 ...
    gdgt23ratio_mean       (lon, lat) float64 ...
    gdgt23ratio_std        (lon, lat) float64 ...
    modernWaterDepth_mean  (lon, lat) float64 ...
Attributes:
    title:             Gridded coretop TEX86 data
    long_name:         statistics of TEX86 for each grid cell with the same l...
    units:             TEX86 [unitless], Scaled RI [unitless]
    data_sources:      Global Ocean Biogeochemistry Hindcast (CMEMS); World O...
    processing_level:  Derived product
    history:           Computed from the global compilation of core-top GDGT ...
    references:        This study

In [9]:
coretop_ds_merged = xr.merge([coretop_ds, ocean_properties_ds])
coretop_da = coretop_ds_merged.to_dataframe().dropna(subset=['scaledRI_median','t_sf2tc_avg']).reset_index()
coretop_da

,lat,lon,region_ID,tex_count,tex_median,tex_mean,tex_std,scaledRI_count,scaledRI_median,scaledRI_mean,...,gdgt23ratio_count,gdgt23ratio_median,gdgt23ratio_mean,gdgt23ratio_std,modernWaterDepth_mean,no3_sf2tc_avg,no3_tc,t_sf2tc_avg,tc_depth,t_tc
0,-77.875,-40.375,46.0,1.0,0.413650,0.413650,NaN,1.0,0.509982,0.509982,...,1.0,1.502839,1.502839,NaN,928.0,NaN,NaN,-1.636017,30.0,-1.75611
1,-77.375,-158.375,32.0,1.0,0.351000,0.351000,NaN,1.0,0.438075,0.438075,...,1.0,2.833333,2.833333,NaN,1081.0,NaN,NaN,-1.586854,50.0,-1.66971
2,-77.125,-45.375,46.0,1.0,0.367016,0.367016,NaN,1.0,0.410931,0.410931,...,1.0,1.824832,1.824832,NaN,332.0,NaN,NaN,-1.692642,30.0,-1.76631
3,-76.625,-35.375,46.0,1.0,0.335974,0.335974,NaN,1.0,0.433342,0.433342,...,1.0,2.066396,2.066396,NaN,932.0,28.365620,28.426125,-1.498910,10.0,-1.56971
4,-76.375,-33.875,46.0,1.0,0.342342,0.342342,NaN,1.0,0.447518,0.447518,...,1.0,2.040640,2.040640,NaN,839.0,29.118933,29.757280,-1.651191,40.0,-1.75551
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1474,84.875,40.625,3.0,1.0,0.334000,0.334000,NaN,1.0,0.459150,0.459150,...,1.0,4.516129,4.516129,NaN,4018.0,7.665815,9.033155,-1.373924,125.0,0.08159
1475,86.625,152.125,3.0,1.0,0.490000,0.490000,NaN,1.0,0.382600,0.382600,...,1.0,21.272727,21.272727,NaN,1467.0,7.059811,9.083502,-1.156047,175.0,0.24909
1476,86.875,-146.125,3.0,1.0,0.438000,0.438000,NaN,1.0,0.399850,0.399850,...,1.0,9.555556,9.555556,NaN,3368.0,8.384475,9.424562,-1.477433,150.0,-0.88321
1477,87.125,104.625,3.0,1.0,0.500000,0.500000,NaN,1.0,0.433075,0.433075,...,1.0,2.870370,2.870370,NaN,4443.0,6.828367,8.589128,-1.377392,150.0,-0.05211


In [10]:
def logistic_fixed_upper(x, x0, k, b):
    return (1-b) / (1 + np.exp(-k * (x - x0))) + b

def logistic(x, x0, k, L, b):
    return L / (1 + np.exp(-k * (x - x0))) + b

In [11]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from scipy.optimize import curve_fit
import plotly.graph_objects as go
import plotly.express as px

def logistic_fixed_upper(x, x0, k, b):
    return (1-b) / (1 + np.exp(-k * (x - x0))) + b

def logistic(x, x0, k, L, b):
    return L / (1 + np.exp(-k * (x - x0))) + b

# ─── prepare sample points ────────────────────────────────────────────────────
xx = np.linspace(-4, 50, 200)

# ─── filter & group your data ──────────────────────────────────────────────────
plot_data = culture_meso_df[culture_meso_df['Temperature'] < 50]
min_count = 2
groups = [g for g, df in plot_data.groupby('Strains_edited') if len(df) >= min_count]

# ─── build the figure with one trace per strain per model ─────────────────────
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=coretop_da['t_sf2tc_avg'],
    y=coretop_da['scaledRI_median'],
    mode='markers',
    marker=dict(color='rgba(178,178,178,1.000)', size=5),
    name='Core-top data',
    showlegend=True   # you can allow toggling it via legend if you like
))
core_idx = 0  # index of this core-top trace

# ─── add global fits ────────────────────────────────────────────────
X_all = plot_data['Temperature'].values.reshape(-1,1)
y_all = plot_data['scaledRI'].values

# -- Linear fit --
ols_all = LinearRegression().fit(X_all, y_all)
y_lin_all = ols_all.predict(xx.reshape(-1,1))


# -- Logistic fixed upper --
p0f = [25, 0.1, 0.1]
bf = (0, [50,1,1])
popt_f_all, _ = curve_fit(logistic_fixed_upper, plot_data['Temperature'], plot_data['scaledRI'], p0=p0f, bounds=bf)
y_logif_all = logistic_fixed_upper(xx, *popt_f_all)
r2_logif_all = 1 - np.var(plot_data['scaledRI'] - logistic_fixed_upper(plot_data['Temperature'], *popt_f_all)) / np.var(plot_data['scaledRI'])

# -- Full logistic --
p0_all = [25,0.1,0.1,0.1]
bnd_all = (0, [50,1,1,1])
popt_all, _ = curve_fit(logistic, plot_data['Temperature'], plot_data['scaledRI'], p0=p0_all, bounds=bnd_all)
y_logi_all = logistic(xx, *popt_all)
r2_logi_all = 1 - np.var(plot_data['scaledRI'] - logistic(plot_data['Temperature'], *popt_all)) / np.var(plot_data['scaledRI'])

## add global fits to figure
for model, y, r2 in [
    ("Linear", y_lin_all, ols_all.score(X_all, y_all)),
    ("Logistic fixed", y_logif_all, r2_logif_all),
    ("Logistic", y_logi_all, r2_logi_all)
]:
    fig.add_trace(go.Scatter(
        x=xx, y=y,
        mode='lines',
        name=f"glob_cul_meso -- {model} (R²={r2:.2f})",
        visible=True
    ))

### add X_all and y_all with coretop data and fit the glob_cul_meso_coretop
X_all_full = np.concatenate((coretop_da['t_sf2tc_avg'].values.reshape(-1,1), plot_data['Temperature'].values.reshape(-1,1)))
y_all_full = np.concatenate((coretop_da['scaledRI_median'].values, plot_data['scaledRI'].values))

# -- Linear fit --
ols_all_full = LinearRegression().fit(X_all_full, y_all_full)
y_lin_all_full = ols_all_full.predict(xx.reshape(-1,1))
# -- Logistic fixed upper --
p0f_full = [25, 0.1, 0.1]
bf_full = (0, [50,1,1])
popt_f_all_full, _ = curve_fit(logistic_fixed_upper, np.squeeze(X_all_full), y_all_full, p0=p0f_full, bounds=bf_full)
y_logif_all_full = logistic_fixed_upper(xx, *popt_f_all_full)
r2_logif_all_full = 1 - np.var(y_all_full - logistic_fixed_upper(X_all_full, *popt_f_all_full)) / np.var(y_all_full)
# -- Full logistic --
p0_all_full = [25,0.1,0.1,0.1]
bnd_all_full = (0, [50,1,1,1])
popt_all_full, _ = curve_fit(logistic, np.squeeze(X_all_full), y_all_full, p0=p0_all_full, bounds=bnd_all_full)
y_logi_all_full = logistic(xx, *popt_all_full)
r2_logi_all_full = 1 - np.var(y_all_full - logistic(X_all_full, *popt_all_full)) / np.var(y_all_full)
## add global fits to figure
for model, y, r2 in [
    ("Linear", y_lin_all_full, ols_all_full.score(X_all_full, y_all_full)),
    ("Logistic fixed", y_logif_all_full, r2_logif_all_full),
    ("Logistic", y_logi_all_full, r2_logi_all_full)
]:
    fig.add_trace(go.Scatter(
        x=xx, y=y,
        mode='lines',
        name=f"glob_cul_meso_coretop -- {model} (R²={r2:.2f})",
        visible=True
    ))
    
# ─── add all your strain traces ───────────────────────────
for strain in groups:
    df = plot_data[plot_data['Strains_edited'] == strain]
    X = df['Temperature'].values.reshape(-1,1)
    y = df['scaledRI'].values
    
    # -- Linear fit --
    ols = LinearRegression().fit(X, y)
    y_lin = ols.predict(xx.reshape(-1,1))
    fig.add_trace(go.Scatter(
        x=xx, y=y_lin, 
        mode='lines',
        name=f"{strain} -- Linear (R²={ols.score(X,y):.2f})",
        visible=(strain == groups[0])  # only first strain visible initially
    ))
    
    # -- Logistic fixed upper --
    p0f = [25, 0.1, 0.1]
    bf = (0, [50,1,1])
    popt_f, _ = curve_fit(logistic_fixed_upper, df['Temperature'], df['scaledRI'], p0=p0f, bounds=bf)
    y_logf = logistic_fixed_upper(xx, *popt_f)
    r2_f = 1 - np.var(df['scaledRI'] - logistic_fixed_upper(df['Temperature'], *popt_f)) / np.var(df['scaledRI'])
    fig.add_trace(go.Scatter(
        x=xx, y=y_logf,
        mode='lines',
        name=f"{strain} -- Logistic fixed (R²={r2_f:.2f})",
        visible=False
    ))
    
    # -- Full logistic --
    p0 = [25,0.1,0.1,0.1]
    bnd = (0, [50,1,1,1])
    popt, _ = curve_fit(logistic, df['Temperature'], df['scaledRI'], p0=p0, bounds=bnd)
    y_log = logistic(xx, *popt)
    r2 = 1 - np.var(df['scaledRI'] - logistic(df['Temperature'], *popt)) / np.var(df['scaledRI'])
    fig.add_trace(go.Scatter(
        x=xx, y=y_log,
        mode='lines',
        name=f"{strain} -- Logistic (R²={r2:.2f})",
        visible=False
    ))
    
    # -- Always-on scatter of raw data --
    fig.add_trace(go.Scatter(
        x=df['Temperature'], y=df['scaledRI'],
        mode='markers',
        marker=dict(size=6, line=dict(width=1, color='black')),
        name=f"{strain} -- data",
        visible=True
    ))
# build a mapping strain → color
palette1 = px.colors.qualitative.Set1
palette2 = px.colors.qualitative.Set2

# make one long list
all_colors = palette1 + palette2

# then map strains in order
strain_colors = {
    strain: all_colors[i % len(all_colors)]
    for i, strain in enumerate(groups)
}

# now loop through all traces
for trace in fig.data:
    # The “cultures + mesocosm only” global fit
    if trace.name.startswith("glob_cul_meso --"):
        trace.update(line=dict(
            color="black",   # pick any CSS name or hex code
            dash="solid",        # “solid”, “dash”, “dot”, “dashdot”, etc
            width=4              # thicker line
        ))

    # The “cultures + mesocosm + coretop” global fit
    elif trace.name.startswith("glob_cul_meso_coretop --"):
        trace.update(line=dict(
            color="black",
            dash="dashdot",
            width=3
        ))
        
    # skip the core-top (and any other truly global or background traces)
    elif trace.name == "Core-top data":
        continue

    # split off the strain name (everything before the “ – ”)
    strain = trace.name.split(" -- ")[0]

    # pick a color: if it’s one of our strains, use its palette color;
    # otherwise (e.g. “Global”) fall back to black or whatever you like
    col = strain_colors.get(strain, "black")

    # apply to both marker & line
    trace.update(
        marker=dict(color=col),
        line=dict(color=col)
    )



# Hide all line traces (fits) by default, leave markers visible
for trace in fig.data:
    if "lines" in trace.mode:
        trace.visible = "legendonly"
    else:
        trace.visible = True

# # Make “raw data” the initial visibility state:
# for tr in fig.data:
#     if tr.name == "Core-top data" or tr.name.endswith("-- data"):
#         tr.visible = True
#     else:
#         tr.visible = False
        


# reposition the model‐dropdown as a vertical nav bar on the left
fig.update_layout(

    autosize=False,     # turn off responsive “fill the container”
    width=900,          # fixed pixel width
    height=500,         # fixed pixel height
    margin=dict(l=40, r=40, t=50, b=40),

    title="Scaled RI vs Temperature",
    xaxis_title="Temperature (°C)",
    yaxis_title="Scaled RI",
    
    yaxis=dict(
        range=[0.1, 1.1],
        autorange=False    # turn off auto‐scaling
    ),

    # put the legend also on the left, so users can click each strain on/off
    legend=dict(
        traceorder="normal",
        orientation="v",
        x=1.13,      # just inside the left margin
        y=0.5,
        xanchor="left",
        yanchor="middle",
        itemsizing="constant",
        bordercolor="black",
        borderwidth=1
    ),
    template='seaborn',
    font=dict(
        family="TeX Gyre Heros",
        size=14,
        color="black"
    ),
    
)

# lock X/Y so the units match (remove if you don’t want a 1:1 scale)
fig.update_xaxes(constrain='domain')            # keep the X‐axis from expanding

# set a default
# fig.update_layout(template="plotly_white")

fig.show()

# in your Python script / notebook:
fig.write_html(
  os.path.join(local_github_path, 'html_figures', 'scaledRI_vs_Temperature.html'),
  include_plotlyjs="cdn",   # pull Plotly.js from the CDN
  full_html=False           # only the <div>+<script>—no <html>/<body> wrapper
)


In [12]:
def get_posteriors(data_dict, stan_file_name):
    '''
    stan_files options:
        - 'joint_culture_meso.stan'
        - 'joint_culture_meso_coretop.stan'
        - 'joint_culture_meso_coretop_multivariate.stan'
        - 'joint_culture_meso_coretop_multivariate_fixedbeta1.stan'
        - 'logistic_fixed_asymptote.stan'
        - 'logistic_fixed_asymptote_seq.stan'
    '''
    from cmdstanpy import CmdStanModel
    stan_models_path = local_github_path+'/stan_models/'
    
    model = CmdStanModel(stan_file=os.path.join(stan_models_path, stan_file_name))
    
    fit = model.sample(
        data=data_dict,
        chains=4, 
        iter_warmup=500, 
        iter_sampling=1000, 
        seed=42,
        parallel_chains=4,
        show_console=True
    )
    
    ### build the posterior xarray dataset
    variable_names = fit.stan_variables().keys()
    
    # create empty xarray dataset
    posterior_ds = xr.Dataset()
    
    for key in variable_names:
        # get the variable data
        variable_data = fit.stan_variable(key)
        
        # create a DataArray for the variable
        da = xr.DataArray(variable_data, dims='draw',
                          name=key,)
        
        # add the DataArray to the dataset
        posterior_ds[key] = da
        
    return posterior_ds
    
    

In [13]:
coretop_da.columns

Index(['lat', 'lon', 'region_ID', 'tex_count', 'tex_median', 'tex_mean',
       'tex_std', 'scaledRI_count', 'scaledRI_median', 'scaledRI_mean',
       'scaledRI_std', 'gdgt23ratio_count', 'gdgt23ratio_median',
       'gdgt23ratio_mean', 'gdgt23ratio_std', 'modernWaterDepth_mean',
       'no3_sf2tc_avg', 'no3_tc', 't_sf2tc_avg', 'tc_depth', 't_tc'],
      dtype='object')

In [14]:
culture_data_full = culture_meso_df[culture_meso_df['Temperature']<50][culture_meso_df['datatype']=='culture'].reset_index(drop=True)
meso_data_full = culture_meso_df[culture_meso_df['Temperature']<50][culture_meso_df['datatype']=='mesocosm'].reset_index(drop=True)
coretop_da

x1 = culture_data_full['Temperature'].astype(float).tolist() 
y1 = culture_data_full['scaledRI'].astype(float).tolist() 

x2 = meso_data_full['Temperature'].astype(float).tolist()
y2 = meso_data_full['scaledRI'].astype(float).tolist()

x3 = coretop_da['t_sf2tc_avg'].astype(float).tolist()
y3 = coretop_da['scaledRI_median'].astype(float).tolist()
# Save data for Stan
data = {
    "N1": len(culture_data_full),
    "x1": x1,
    "y1": y1,
    "N2": len(meso_data_full),
    "x2": x2,
    "y2": y2,
}

stan_file_name = 'joint_culture_meso.stan'
joint_cul_meso_post_ds = get_posteriors(data, stan_file_name)
joint_cul_meso_post_ds


### cul + meso + coretop
data = {
    "N1": len(culture_data_full),
    "x1": x1,
    "y1": y1,
    "N2": len(meso_data_full),
    "x2": x2,
    "y2": y2,
    "N3": len(coretop_da),
    "x3": x3,
    "y3": y3,

}
stan_file_name = 'joint_culture_meso_coretop.stan'
joint_cul_meso_coretop_post_ds = get_posteriors(data, stan_file_name)
joint_cul_meso_coretop_post_ds


14:18:56 - cmdstanpy - INFO - Chain [1] start processing
14:18:56 - cmdstanpy - INFO - Chain [2] start processing
14:18:56 - cmdstanpy - INFO - Chain [3] start processing
14:18:56 - cmdstanpy - INFO - Chain [4] start processing


Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 1000 (Default)
Chain [1] num_warmup = 500
Chain [1] save_warmup = false (Default)
Chain [1] thin = 1 (Default)
Chain [1] adapt
Chain [1] engaged = true (Default)
Chain [1] gamma = 0.05 (Default)
Chain [1] delta = 0.8 (Default)
Chain [1] kappa = 0.75 (Default)
Chain [1] t0 = 10 (Default)
Chain [1] init_buffer = 75 (Default)
Chain [1] term_buffer = 50 (Default)
Chain [1] window = 25 (Default)
Chain [1] save_metric = false (Default)
Chain [1] algorithm = hmc (Default)
Chain [1] hmc
Chain [1] engine = nuts (Default)
Chain [1] nuts
Chain [1] max_depth = 10 (Default)
Chain [1] metric = diag_e (Default)
Chain [1] metric_file =  (Default)
Chain [1] stepsize = 1 (Default)
Chain [1] stepsize_jitter = 0 (Default)
Chain [1] num_chains = 1 (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /tmp/tmpz7r6visb/0shrynlg.json
Chain [1] init = 2 (Default)
Chain [1] random
Chain [1] seed = 42
Chain [1] output
Cha

14:18:57 - cmdstanpy - INFO - Chain [1] done processing
14:18:57 - cmdstanpy - INFO - Chain [2] done processing


Chain [4] Iteration: 1000 / 1500 [ 66%]  (Sampling)
Chain [1] Iteration: 1400 / 1500 [ 93%]  (Sampling)
Chain [2] Iteration: 1200 / 1500 [ 80%]  (Sampling)
Chain [3] Iteration:  900 / 1500 [ 60%]  (Sampling)
Chain [1] Iteration: 1500 / 1500 [100%]  (Sampling)
Chain [1] 
Chain [1] Elapsed Time: 0.254 seconds (Warm-up)
Chain [1] 0.462 seconds (Sampling)
Chain [1] 0.716 seconds (Total)
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1]

14:18:57 - cmdstanpy - INFO - Chain [4] done processing
14:18:57 - cmdstanpy - INFO - Chain [3] done processing
14:18:57 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'joint_culture_meso.stan', line 45, column 2 to column 27)


...

14:19:06 - cmdstanpy - INFO - Chain [1] done processing
14:19:06 - cmdstanpy - INFO - Chain [4] done processing


Chain [1] Iteration: 1500 / 1500 [100%]  (Sampling)
Chain [1] 
Chain [1] Elapsed Time: 3.407 seconds (Warm-up)
Chain [1] 4.61 seconds (Sampling)
Chain [1] 8.017 seconds (Total)
Chain [1] 
Chain [1] 
Chain [4] Iteration: 1500 / 1500 [100%]  (Sampling)
Chain [4] 
Chain [4] Elapsed Time: 3.14 seconds (Warm-up)
Chain [4] 4.909 seconds (Sampling)
Chain [4] 8.049 seconds (Total)
Chain [4] 
Chain [4] 
Chain [2] Iteration: 1200 / 1500 [ 80%]  (Sampling)
Chain [2] Iteration: 1300 / 1500 [ 86%]  (Sampling)
Chain [2] Iteration: 1400 / 1500 [ 93%]  (Sampling)


14:19:07 - cmdstanpy - INFO - Chain [2] done processing
14:19:07 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'joint_culture_meso_coretop.stan', line 46, column 2 to column 27)
	Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'joint_culture_meso_coretop.stan', line 46, column 2 to column 27)
	Exception: normal_lpdf: Location parameter[1] is nan, but must be finite! (in 'joint_culture_meso_coretop.stan', line 44, column 2 to column 27)
Exception: normal_lpdf: Location parameter[1] is nan, but must be finite! (in 'joint_culture_meso_coretop.stan', line 44, column 2 to column 27)
Exception: normal_lpdf: Location parameter[1] is nan, but must be finite! (in 'joint_culture_meso_coretop.stan', line 44, column 2 to column 27)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'joint_culture_meso_coretop.stan', line 46, column 2 to column 27)
	Exception: normal_lpdf:

Chain [2] Iteration: 1500 / 1500 [100%]  (Sampling)
Chain [2] 
Chain [2] Elapsed Time: 5.277 seconds (Warm-up)
Chain [2] 3.856 seconds (Sampling)
Chain [2] 9.133 seconds (Total)
Chain [2] 
Chain [2] 


<xarray.Dataset>
Dimensions:        (draw: 4000)
Dimensions without coordinates: draw
Data variables:
    x0             (draw) float64 28.34 28.11 28.33 28.15 ... 28.5 28.46 28.05
    k              (draw) float64 0.1377 0.1407 0.1363 ... 0.1288 0.1293 0.1277
    b              (draw) float64 0.4426 0.4392 0.4404 ... 0.4347 0.4372 0.4304
    sigma1         (draw) float64 0.09188 0.08622 0.09605 ... 0.09239 0.09647
    sigma2         (draw) float64 0.0596 0.07938 0.06874 ... 0.06065 0.08518
    sigma3         (draw) float64 0.06025 0.0586 0.05886 ... 0.05603 0.06328
    sigma2_sigma1  (draw) float64 0.6487 0.9206 0.7157 ... 0.6002 0.6564 0.883
    sigma3_sigma1  (draw) float64 0.6558 0.6797 0.6128 ... 0.6417 0.6065 0.656

In [95]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import FloatRangeSlider, VBox
from IPython.display import display

### plot data
culture_data_full = culture_meso_df[culture_meso_df['Temperature']<50][culture_meso_df['datatype']=='culture'].reset_index(drop=True)
meso_data_full = culture_meso_df[culture_meso_df['Temperature']<50][culture_meso_df['datatype']=='mesocosm'].reset_index(drop=True)
coretop_da

# Set seed
np.random.seed(42)

fig = make_subplots(
    rows=2,
    cols=4,
    specs=[
        [{"colspan": 2}, None, {"colspan": 2}, None],   # First row: 1 plot spanning all 3 columns
        [{}, {}, {}, {}]                    # Second row: normal 3 plots
    ],
    vertical_spacing=0.1,
    row_heights=[1, 0.5],
    subplot_titles=("Scatter", "Small Plot 1", "Small Plot 2", "Small Plot 3")
)

### plot scatter plot in subplot 1

### plot core-top data
fig.add_trace(
    go.Scatter(
        x=coretop_da['t_sf2tc_avg'],
        y=coretop_da['scaledRI_median'],
        mode='markers',
        marker=dict(color='lightgray', size=5),
        name='Core-top data',
        showlegend=True
    ),
    row=1, col=1
)
### plot culture data
fig.add_trace(
    go.Scatter(
        x=culture_data_full['Temperature'],
        y=culture_data_full['scaledRI'],
        mode='markers',
        marker=dict(color='blue', size=6, line=dict(width=1, color='black')),
        name='Culture data',
        showlegend=True
    ),
    row=1, col=1
)

### plot mesocosm data
fig.add_trace(
    go.Scatter(
        x=meso_data_full['Temperature'],
        y=meso_data_full['scaledRI'],
        mode='markers',
        marker=dict(color='orange', size=6, line=dict(width=1, color='black')),
        name='Mesocosm data',
        showlegend=True
    ),
    row=1, col=1
)

# Add overlaid histograms
x0_prior0 = np.random.normal(20, 20, 1000)
x0_post_joint_cul_meso = joint_cul_meso_post_ds['x0'].values
x0_post_joint_cul_meso_coretop = joint_cul_meso_coretop_post_ds['x0'].values
plot_data_list = [
    (x0_prior0, 'x0 prior', 'gray'),
    (x0_post_joint_cul_meso, 'x0 post (joint-cul-meso)', 'blue'),
    (x0_post_joint_cul_meso_coretop, 'x0 post (joint-cul-meso-coretop)', 'orange')
]
for data, label, color in plot_data_list:
    fig.add_trace(
        go.Histogram(
            x=data,
            name=label,
            histnorm='probability density',
            marker_color=color,
            opacity=0.5,
            xbins=dict(size=1)
        ),
        row=2, col=1
    )

k_prior0 = np.random.normal(0, 0.25, 1000)
k_post_joint_cul_meso = joint_cul_meso_post_ds['k'].values
k_post_joint_cul_meso_coretop = joint_cul_meso_coretop_post_ds['k'].values
plot_data_list = [
    (k_prior0, 'k prior', 'gray'),
    (k_post_joint_cul_meso, 'k posterior', 'blue'),
    (k_post_joint_cul_meso_coretop, 'k posterior (joint-cul-meso-coretop)', 'orange')
]
for data, label, color in plot_data_list:
    fig.add_trace(
        go.Histogram(
            x=data,
            name=label,
            histnorm='probability density',
            marker_color=color,
            opacity=0.5,
            xbins=dict(size=0.01)
        ),
        row=2, col=2
    )

b_prior0 = np.random.beta(2,5,1000)
b_post_joint_cul_meso = joint_cul_meso_post_ds['b'].values
b_post_joint_cul_meso_coretop = joint_cul_meso_coretop_post_ds['b'].values

plot_data_list = [
    (b_prior0, 'b prior', 'gray'),
    (b_post_joint_cul_meso, 'b posterior', 'blue'),
    (b_post_joint_cul_meso_coretop, 'b posterior (joint-cul-meso-coretop)', 'orange')
]

for data, label, color in plot_data_list:
    fig.add_trace(
        go.Histogram(
            x=data,
            name=label,
            histnorm='probability density',
            marker_color=color,
            opacity=0.5,
            xbins=dict(size=0.01)
        ),
        row=2, col=3
    )

fig.update_layout(
    barmode='overlay',
    title="Bayesian posterior distributions",
    height=800,
    width=1200,
    template='seaborn',
    font=dict(family="TeX Gyre Heros", size=14, color="black"),
    legend=dict(
        orientation="v",
        y=0.5,
        x=1,
        xanchor='left',
        yanchor='middle',
    )
)

fig.show()

In [82]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=2,
    cols=4,
    specs=[
        [{"colspan": 2}, None, {"colspan": 2}, None],   # First row: 1 plot spanning all 3 columns
        [{}, {}, {}, {}]                    # Second row: normal 3 plots
    ],
    vertical_spacing=0.1,
    row_heights=[1, 0.5],
    subplot_titles=("Big Plot", "Small Plot 1", "Small Plot 2", "Small Plot 3",'test')
)

# Add traces
fig.add_trace(go.Scatter(y=[1,2,3]), row=1, col=1)  # Big plot
fig.add_trace(go.Scatter(y=[3,2,1]), row=1, col=3)  # Small plot 1
fig.add_trace(go.Scatter(y=[2,3,1]), row=2, col=2)  # Small plot 2
fig.add_trace(go.Scatter(y=[1,1,1]), row=2, col=3)  # Small plot 3

### make the second subplot "legendonly"

fig.data[1].visible = 'legendonly'

fig.update_layout(height=800, width=800, title_text="Custom Subplot Layout")
fig.update_yaxes(
    scaleanchor="x",  # LOCK y-axis scale to x-axis for the first subplot
    row=1, col=1
)
fig.show()


In [83]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create subplot layout
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=("Plot 1", "Plot 2", "Plot 3", "Plot 4"),
    vertical_spacing=0.15,
    horizontal_spacing=0.15
)

# Example data
colors = ["blue", "red"]

# Add traces
for row in range(1, 3):
    for col in range(1, 3):
        fig.add_trace(go.Scatter(x=[1,2,3], y=[row+col, row+col+1, row+col+2], mode='lines',
                                 name=f"Data 1", line=dict(color="blue")), row=row, col=col)
        fig.add_trace(go.Scatter(x=[1,2,3], y=[row+col+2, row+col+1, row+col], mode='lines',
                                 name=f"Data 2", line=dict(color="red")), row=row, col=col)

# Hide the global legend
fig.update_layout(showlegend=False)

# Now add mini legends manually using annotations
annotations = []

# Define mini legend positions (you can tweak)
legend_positions = {
    (1, 1): (0.18, 1.0),
    (1, 2): (0.85, 1.0),
    (2, 1): (0.18, 0.45),
    (2, 2): (0.85, 0.45)
}

for (row, col), (xpos, ypos) in legend_positions.items():
    annotations += [
        # First item
        dict(x=xpos, y=ypos, xref="paper", yref="paper",
             text=f"<span style='color:blue'>●</span> Data 1", showarrow=False, font=dict(size=12)),
        # Second item
        dict(x=xpos, y=ypos-0.04, xref="paper", yref="paper",
             text=f"<span style='color:red'>●</span> Data 2", showarrow=False, font=dict(size=12))
    ]

fig.update_layout(
    height=700,
    width=900,
    title_text="Subplots with Mini Legends",
    annotations=annotations
)

fig.show()
